## Acknowledgements

To start, I’d like to thank **Tong Hui Kang** and **konbu17**.

The CoT data and part of the hyperparameter settings used in this notebook are adapted from Tong Hui Kang’s open-source GitHub repository, while the training code is modified based on konbu17’s public notebook.  
- [Tong Hui Kang’s open-source GitHub repository](https://github.com/tonghuikang/nemotron)
- [konbu17’s public notebook](https://www.kaggle.com/code/konbu17/nemotron-sft-lora-with-cot)

## Overview

In this notebook, I will try to reproduce the method released by Tong Hui Kang and train a model that can reach around **0.85** on the public leaderboard.

However, due to differences in the training platform, I can only make my setup as close as possible to Tong Hui Kang’s original configuration. In addition, I did not use the generated augmentation data in this reproduction. The scripts for generating such augmented data can be found in Tong Hui Kang’s repository.

As a result, the final score of this notebook is expected to be slightly lower than the score of Tong Hui Kang’s currently public model.

## Notes on Training

Also, I am still a beginner in practical LLM training, so there are likely many inefficiencies in my training setup.

Although I tried to speed up the process with **Unsloth**, I only managed to reduce the training time from **10+ hours** to **7+ hours**. By comparison, Tong Hui Kang mentioned when sharing his method publicly that even without using **Tinker**, each of his training runs only took **4+ hours**.

Even so, being able to train a model like this by myself was still a rewarding experience. I hope this notebook can also help others participate more effectively in this competition.

## Mode Selection

In [1]:
# ============================================================
# MODE SELECTION — set exactly one to 1
# ============================================================

import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

# Mode A: Train LoRA from scratch on Kaggle GPU
TRAIN_ON_KAGGLE = 1

# Mode B: Use pre-trained LoRA weights from dataset and just package them
USE_PRETRAINED = 0

assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1, \
    "Set exactly one of TRAIN_ON_KAGGLE / USE_PRETRAINED to 1."

PRETRAINED_ADAPTER_DATASET_PATH = "/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection"
BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

print({
    "TRAIN_ON_KAGGLE": TRAIN_ON_KAGGLE,
    "USE_PRETRAINED": USE_PRETRAINED,
    "PRETRAINED_ADAPTER_DATASET_PATH": PRETRAINED_ADAPTER_DATASET_PATH,
})


{'TRAIN_ON_KAGGLE': 1, 'USE_PRETRAINED': 0, 'PRETRAINED_ADAPTER_DATASET_PATH': '/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection'}


## Setup & Model Loading

In [ ]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)

if candidates:
    wheel = candidates[0]
    target = "/kaggle/working/pydeps"
    os.makedirs(target, exist_ok=True)

    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--no-deps",
            "--target", target,
            "--upgrade",
            "--ignore-installed",
            wheel,
        ],
        check=True,
    )

    if target not in sys.path:
        sys.path.insert(0, target)
    site.addsitedir(target)

    import importlib.util
    print("Custom target added:", target)
    print("triton spec:", importlib.util.find_spec("triton"))
else:
    print("No offline Triton wheel found. Continuing with the runtime-provided Triton.")



In [3]:
if TRAIN_ON_KAGGLE:
    import sys, os, shutil, stat

    # Add utility script to Python path (provides helper binaries)
    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')

    # Copy ptxas-blackwell to /tmp with execute permissions
    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst

        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst

    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'

    print('Training environment fixes applied.')
else:
    print("USE_PRETRAINED=1: skipping Triton / ptxas environment fixes.")


Training environment fixes applied.


In [ ]:
from pathlib import Path
import random

DEFAULT_MANIFEST_CSV_CANDIDATES = [
    Path("/kaggle/input/winning_snapshot_delta_manifest/winning_snapshot_delta_manifest.csv"),
    Path("/kaggle/input/winning_snapshot_delta_manifest/manifest.csv"),
    Path("/kaggle/input/winning-snapshot-delta-manifest/winning_snapshot_delta_manifest.csv"),
    Path("/kaggle/input/winning-snapshot-delta-manifest/manifest.csv"),
    Path("/kaggle/input/nemotron-training-manifest/winning_snapshot_delta_manifest.csv"),
    Path("/kaggle/input/nemotron-training-manifest/manifest.csv"),
]

SEED = 123
EPOCH_BATCH_ORDER_SEED = 0
TRAINING_MANIFEST_CSV_PATH = next((str(p) for p in DEFAULT_MANIFEST_CSV_CANDIDATES if p.exists()), None)
MANIFEST_ONLY_MODE = True
MAX_SEQ_LEN = 8192
REASONING_TOKEN_BUDGET = 7680
NUM_EPOCHS = 1
EFFECTIVE_BATCH_SIZE = 32
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 32
LEARNING_RATE = 2e-4
LOSS_CONFIG_NAME = "cross_entropy"
LR_SCHEDULE_NAME = "step_linear_decay"
USE_4BIT = True
RUN_LOGPROB_ANALYSIS = True
LOGPROB_ANALYSIS_LIMIT = None
TRAINING_LOGS_ROOT = "/kaggle/working/training/sft"

if TRAINING_MANIFEST_CSV_PATH is None:
    raise FileNotFoundError(
        "Could not find a training manifest CSV in Kaggle inputs. "
        "Attach the winning_snapshot_delta_manifest dataset or set TRAINING_MANIFEST_CSV_PATH manually."
    )

random.seed(SEED)
print({
    "MANIFEST_ONLY_MODE": MANIFEST_ONLY_MODE,
    "SEED": SEED,
    "EPOCH_BATCH_ORDER_SEED": EPOCH_BATCH_ORDER_SEED,
    "TRAINING_MANIFEST_CSV_PATH": TRAINING_MANIFEST_CSV_PATH,
    "MAX_SEQ_LEN": MAX_SEQ_LEN,
    "REASONING_TOKEN_BUDGET": REASONING_TOKEN_BUDGET,
    "NUM_EPOCHS": NUM_EPOCHS,
    "EFFECTIVE_BATCH_SIZE": EFFECTIVE_BATCH_SIZE,
    "PER_DEVICE_BATCH_SIZE": PER_DEVICE_BATCH_SIZE,
    "GRADIENT_ACCUMULATION_STEPS": GRADIENT_ACCUMULATION_STEPS,
    "LEARNING_RATE": LEARNING_RATE,
    "LOSS_CONFIG_NAME": LOSS_CONFIG_NAME,
    "LR_SCHEDULE_NAME": LR_SCHEDULE_NAME,
    "USE_4BIT": USE_4BIT,
    "RUN_LOGPROB_ANALYSIS": RUN_LOGPROB_ANALYSIS,
    "LOGPROB_ANALYSIS_LIMIT": LOGPROB_ANALYSIS_LIMIT,
    "TRAINING_LOGS_ROOT": TRAINING_LOGS_ROOT,
})


In [ ]:
if TRAIN_ON_KAGGLE:
    import glob
    import os
    import subprocess
    import sys

    def recursive_wheels(pattern: str):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    all_mamba = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")

    print("Found mamba wheels:", all_mamba)
    print("Found causal-conv1d wheels:", all_causal)

    import torch
    print("Python:", sys.version)
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Torch CUDA:", torch.version.cuda)

    if not torch.cuda.is_available():
        raise RuntimeError("TRAIN_ON_KAGGLE=1 requires a GPU runtime.")

    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")

    base_packages = [
        "transformers",
        "datasets",
        "accelerate",
        "peft",
        "bitsandbytes",
        "tokenizers",
        "sentencepiece",
        "safetensors",
    ]
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-index",
            "--find-links",
            packages_dir,
            *base_packages,
        ],
        check=True,
    )

    def pick_last(wheels):
        return wheels[-1] if wheels else None

    causal_wheel = pick_last(all_causal)
    mamba_wheel = pick_last(all_mamba)
    print("Selected causal wheel:", causal_wheel)
    print("Selected mamba wheel:", mamba_wheel)

    if causal_wheel:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel],
            check=True,
        )
    if mamba_wheel:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel],
            check=True,
        )
    else:
        raise FileNotFoundError("Could not find a compatible mamba_ssm wheel under /kaggle/input.")

    print("Offline package installation finished.")
else:
    print("USE_PRETRAINED=1: skipping standard HF / PEFT package installation.")



In [ ]:
if TRAIN_ON_KAGGLE:
    import torch
    import kagglehub
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    load_kwargs = {
        "trust_remote_code": True,
        "attn_implementation": "eager",
        "low_cpu_mem_usage": True,
    }
    if USE_4BIT:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        load_kwargs.update(
            {
                "torch_dtype": torch.bfloat16,
                "quantization_config": bnb_config,
                "device_map": "auto",
            }
        )
    else:
        load_kwargs.update(
            {
                "torch_dtype": torch.bfloat16,
                "device_map": "auto",
            }
        )

    model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, **load_kwargs)
    model.config.use_cache = False
    print("Model loaded with standard Transformers.")
else:
    print("USE_PRETRAINED=1: skipping base model and tokenizer loading.")



In [ ]:
if TRAIN_ON_KAGGLE:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    LORA_RANK = 32
    LORA_ALPHA = 32
    LORA_DROPOUT = 0.0
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "in_proj", "out_proj", "up_proj", "down_proj",
        "lm_head",
    ]

    if USE_4BIT:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    else:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        if hasattr(model, "enable_input_require_grads"):
            model.enable_input_require_grads()

    peft_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=target_modules,
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()
else:
    print("USE_PRETRAINED=1: skipping trainable LoRA construction.")



## Mode A: Train on Kaggle

In [ ]:
if TRAIN_ON_KAGGLE:
    import csv
    import json
    import math
    import os
    import random
    import time
    from collections import Counter, defaultdict
    from datetime import datetime
    from pathlib import Path

    import torch
    import torch.nn.functional as F
    from datasets import Dataset as HFDataset
    from torch.optim.lr_scheduler import LambdaLR
    from torch.utils.data import DataLoader, Sampler
    from transformers import Trainer, TrainingArguments

    class CrossEntropyLossConfig:
        name = "cross_entropy"
        class_name = "CrossEntropyLossConfig"

    class StepLinearDecayLRSchedule:
        class_name = "StepLinearDecayLRSchedule"

        def __init__(self, learning_rate: float = 2e-4):
            self.learning_rate = learning_rate

        def get_lr(self, step: int, total_steps: int, epoch: int, total_epochs: int) -> float:
            return self.learning_rate * (1 - step / total_steps)

    if LOSS_CONFIG_NAME != "cross_entropy":
        raise NotImplementedError(
            "This notebook is pinned to the winning default loss: cross_entropy."
        )
    if LR_SCHEDULE_NAME != "step_linear_decay":
        raise NotImplementedError(
            "This notebook is pinned to the winning default LR schedule: step_linear_decay."
        )
    if EFFECTIVE_BATCH_SIZE != PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS:
        raise ValueError(
            "EFFECTIVE_BATCH_SIZE must equal PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS"
        )

    loss_config = CrossEntropyLossConfig()
    lr_schedule_config = StepLinearDecayLRSchedule(learning_rate=LEARNING_RATE)

    manifest_csv_path = Path(TRAINING_MANIFEST_CSV_PATH)
    if not manifest_csv_path.exists():
        raise FileNotFoundError(f"Training manifest CSV not found: {manifest_csv_path}")

    def load_records_from_manifest(manifest_path: Path):
        manifest_records = []
        with manifest_path.open(newline="") as f:
            for row in csv.DictReader(f):
                input_ids = json.loads(row["input_ids_json"])
                mask = json.loads(row["mask_json"])
                if len(input_ids) != len(mask):
                    raise ValueError(f"Length mismatch in manifest row {row['problem_id']}")
                if len(input_ids) > MAX_SEQ_LEN:
                    raise ValueError(
                        f"Manifest row {row['problem_id']} exceeds max length: {len(input_ids)} > {MAX_SEQ_LEN}"
                    )
                labels = [token if m == 1 else -100 for token, m in zip(input_ids, mask)]
                manifest_records.append(
                    {
                        "problem_id": row["problem_id"],
                        "source_problem_id": row["source_problem_id"],
                        "category": row["category"],
                        "segment": row.get("segment", "synthetic.jsonl"),
                        "num_loss_tokens": int(row["num_loss_tokens"]),
                        "completion_token_count": int(row.get("completion_token_count") or row["num_loss_tokens"]),
                        "input_ids": input_ids,
                        "attention_mask": [1] * len(input_ids),
                        "labels": labels,
                    }
                )
        manifest_records.sort(key=lambda record: record["problem_id"])
        return manifest_records

    records = load_records_from_manifest(manifest_csv_path)
    record_categories = [record["category"] for record in records]
    completion_lengths = [record["completion_token_count"] for record in records]
    total_tokens = sum(len(record["input_ids"]) for record in records)
    total_loss_tokens = sum(record["num_loss_tokens"] for record in records)
    max_completion_tokens = max(completion_lengths, default=0)
    over_budget = sum(length > REASONING_TOKEN_BUDGET for length in completion_lengths)

    print({
        "training_manifest_csv": str(manifest_csv_path),
        "manifest_examples": len(records),
        "manifest_total_tokens": total_tokens,
        "manifest_loss_tokens": total_loss_tokens,
        "max_completion_tokens": max_completion_tokens,
        "completion_over_7680": over_budget,
        "loss_config": loss_config.class_name,
        "lr_schedule": lr_schedule_config.class_name,
    })
    print(f"Training categories: {Counter(record_categories)}")

    train_dataset = HFDataset.from_list(records)

    class MaskedDataCollator:
        def __init__(self, pad_token_id: int):
            self.pad_token_id = pad_token_id

        def __call__(self, features):
            max_len = max(len(feature["input_ids"]) for feature in features)
            input_ids = []
            attention_mask = []
            labels = []
            for feature in features:
                pad_len = max_len - len(feature["input_ids"])
                input_ids.append(feature["input_ids"] + [self.pad_token_id] * pad_len)
                attention_mask.append(feature["attention_mask"] + [0] * pad_len)
                labels.append(feature["labels"] + [-100] * pad_len)
            return {
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
            }

    def build_stratified_index_order(labels, batch_size, seed):
        by_label = defaultdict(list)
        for idx, label in enumerate(labels):
            by_label[label].append(idx)

        rng = random.Random(seed)
        for idx_list in by_label.values():
            rng.shuffle(idx_list)

        n_batches = max(1, math.ceil(len(labels) / batch_size))
        batches = [[] for _ in range(n_batches)]
        batch_order = list(range(n_batches))
        rng.shuffle(batch_order)

        assigned = 0
        for label in sorted(by_label.keys()):
            for idx in by_label[label]:
                batches[batch_order[assigned % n_batches]].append(idx)
                assigned += 1

        order = [idx for batch in batches for idx in batch]
        if len(order) != len(labels):
            raise ValueError("Stratified order size mismatch")
        return order

    class PrecomputedOrderSampler(Sampler):
        def __init__(self, order):
            self.order = list(order)

        def __iter__(self):
            return iter(self.order)

        def __len__(self):
            return len(self.order)

    class StratifiedTrainer(Trainer):
        def __init__(self, *args, stratified_order=None, loss_config=None, lr_schedule_config=None, **kwargs):
            super().__init__(*args, **kwargs)
            self.stratified_order = stratified_order
            self.loss_config = loss_config or CrossEntropyLossConfig()
            self.lr_schedule_config = lr_schedule_config or StepLinearDecayLRSchedule(
                learning_rate=self.args.learning_rate
            )

        def get_train_dataloader(self):
            if self.train_dataset is None:
                raise ValueError("Trainer requires a train_dataset.")
            if self.stratified_order is None:
                return super().get_train_dataloader()
            sampler = PrecomputedOrderSampler(self.stratified_order)
            return DataLoader(
                self.train_dataset,
                batch_size=self.args.per_device_train_batch_size,
                sampler=sampler,
                collate_fn=self.data_collator,
                num_workers=self.args.dataloader_num_workers,
                pin_memory=torch.cuda.is_available(),
            )

        def create_scheduler(self, num_training_steps: int, optimizer=None):
            if self.lr_scheduler is None:
                optimizer = optimizer or self.optimizer
                if optimizer is None:
                    raise ValueError("Trainer optimizer must be created before scheduler.")
                total_steps = max(num_training_steps, 1)
                base_lr = max(self.lr_schedule_config.learning_rate, 1e-12)

                def lr_lambda(current_step: int) -> float:
                    step = min(max(current_step, 0), total_steps)
                    lr = self.lr_schedule_config.get_lr(
                        step=step,
                        total_steps=total_steps,
                        epoch=0,
                        total_epochs=max(int(self.args.num_train_epochs), 1),
                    )
                    return max(lr, 0.0) / base_lr

                self.lr_scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)
            return self.lr_scheduler

        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            if self.loss_config.name != "cross_entropy":
                raise NotImplementedError(
                    "Only the winning default cross_entropy loss is implemented in this notebook."
                )
            outputs = model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
            )
            shift_logits = outputs.logits[:, :-1, :].contiguous()
            shift_labels = inputs["labels"][:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.reshape(-1),
                ignore_index=-100,
            )
            return (loss, outputs) if return_outputs else loss

    def collect_logprob_logs(records_to_analyze, log_root: Path):
        started = time.time()
        logprob_epoch_dir = log_root / "logprobs" / "0"
        tokens_dir = log_root / "tokens"
        logprob_epoch_dir.mkdir(parents=True, exist_ok=True)
        tokens_dir.mkdir(parents=True, exist_ok=True)

        config = {
            "time": log_root.name,
            "model_name": BASE_MODEL_NAME,
            "batch_size": EFFECTIVE_BATCH_SIZE,
            "num_epochs": NUM_EPOCHS,
            "lora_rank": LORA_RANK,
            "max_length": MAX_SEQ_LEN,
            "reasoning_token_budget": REASONING_TOKEN_BUDGET,
            "use_4bit": USE_4BIT,
            "training_manifest_csv": str(manifest_csv_path),
            "loss_config": {"name": loss_config.name, "class_name": loss_config.class_name},
            "lr_schedule": {
                "class_name": lr_schedule_config.class_name,
                "learning_rate": lr_schedule_config.learning_rate,
            },
            "stats": {
                "num_examples": len(records_to_analyze),
                "total_tokens": sum(len(record["input_ids"]) for record in records_to_analyze),
                "total_unmasked_tokens": sum(record["num_loss_tokens"] for record in records_to_analyze),
                "total_masked_tokens": sum(len(record["input_ids"]) - record["num_loss_tokens"] for record in records_to_analyze),
            },
        }
        with (log_root / "config.json").open("w") as f:
            json.dump(config, f, indent=2)

        logpaths_file = log_root.parent / "logpaths.txt"
        existing = set(logpaths_file.read_text().splitlines()) if logpaths_file.exists() else set()
        if log_root.name not in existing:
            with logpaths_file.open("a") as f:
                f.write(log_root.name + "\n")

        device = next(model.parameters()).device
        model.eval()

        cat_loss = defaultdict(float)
        cat_tokens = defaultdict(int)
        cat_min_lp = {}
        all_unmasked_lps = []
        index_records = []

        metrics_path = log_root / "metrics.jsonl"
        loss_path = log_root / "loss.jsonl"
        index_path = log_root / "logprobs" / "index.jsonl"
        worst_path = log_root / "worst_min_logprob.jsonl"

        with metrics_path.open("w") as metrics_file, index_path.open("w") as index_file:
            for idx, record in enumerate(records_to_analyze, start=1):
                input_ids = torch.tensor([record["input_ids"]], dtype=torch.long, device=device)
                attention_mask = torch.tensor([record["attention_mask"]], dtype=torch.long, device=device)

                with torch.inference_mode():
                    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits[:, :-1, :].float()
                    target_ids = input_ids[:, 1:]
                    logprobs = F.log_softmax(logits, dim=-1).gather(-1, target_ids.unsqueeze(-1)).squeeze(-1)[0]

                logprobs_list = [round(v, 4) for v in logprobs.detach().cpu().tolist()]
                target_mask = [1 if label != -100 else 0 for label in record["labels"][1:]]
                unmasked_lps = [lp for lp, m in zip(logprobs_list, target_mask) if m]
                total_loss = round(sum(-lp for lp in unmasked_lps), 4)
                min_logprob = round(min(unmasked_lps), 4) if unmasked_lps else 0.0

                logprob_path = logprob_epoch_dir / record["problem_id"] / record["segment"]
                logprob_path.parent.mkdir(parents=True, exist_ok=True)
                with logprob_path.open("w") as f:
                    f.write(json.dumps({"logprobs": logprobs_list}) + "\n")

                token_path = tokens_dir / record["problem_id"] / f"{record['segment'].removesuffix('.jsonl')}.json"
                token_path.parent.mkdir(parents=True, exist_ok=True)
                with token_path.open("w") as f:
                    json.dump(
                        {
                            "tokens": record["input_ids"],
                            "mask": [1 if label != -100 else 0 for label in record["labels"]],
                        },
                        f,
                    )

                index_record = {
                    "epoch": 0,
                    "step": 0,
                    "problem_id": record["problem_id"],
                    "segment": record["segment"],
                    "category": record["category"],
                    "num_loss_tokens": record["num_loss_tokens"],
                    "total_loss": total_loss,
                    "min_logprob": min_logprob,
                }
                index_file.write(json.dumps(index_record) + "\n")
                index_records.append(index_record)

                cat_loss[record["category"]] += total_loss
                cat_tokens[record["category"]] += record["num_loss_tokens"]
                if record["category"] not in cat_min_lp or min_logprob < cat_min_lp[record["category"]]:
                    cat_min_lp[record["category"]] = min_logprob
                all_unmasked_lps.extend(unmasked_lps)

                if idx % 100 == 0:
                    print(f"Logprob analysis: {idx}/{len(records_to_analyze)}")
                    metrics_file.flush()
                    index_file.flush()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

            elapsed = round(time.time() - started, 2)
            metrics_record = {
                "epoch": 0,
                "step": 0,
                "lr": LEARNING_RATE,
                "n": len(records_to_analyze),
                "elapsed": elapsed,
                "time": datetime.now().strftime("%m-%d-%H-%M"),
            }
            total_loss_tokens_local = sum(cat_tokens.values())
            for cat in sorted(cat_loss):
                if cat_tokens[cat] > 0:
                    metrics_record[f"_loss_per_token/{cat}"] = round(cat_loss[cat] / cat_tokens[cat], 6)
            if total_loss_tokens_local > 0:
                metrics_record["_loss_per_token"] = round(sum(cat_loss.values()) / total_loss_tokens_local, 6)
            for cat in sorted(cat_min_lp):
                metrics_record[f"_min_logprob/{cat}"] = round(cat_min_lp[cat], 4)
            metrics_file.write(json.dumps(metrics_record) + "\n")

        nll_per_token = round(-sum(all_unmasked_lps) / len(all_unmasked_lps), 6) if all_unmasked_lps else 0.0
        perplexity = round(math.exp(min(nll_per_token, 20)), 4) if all_unmasked_lps else 1.0
        with loss_path.open("w") as loss_file:
            loss_file.write(json.dumps({
                "epoch": 0,
                "metrics": [
                    [{"nll_per_token": nll_per_token}],
                    [{"perplexity": perplexity}],
                ],
            }) + "\n")

        with worst_path.open("w") as f:
            for record in sorted(index_records, key=lambda item: item["min_logprob"])[:50]:
                f.write(json.dumps(record) + "\n")

        print(f"Logprob analysis saved to {log_root}")
        print("Worst min_logprob examples:")
        for record in sorted(index_records, key=lambda item: item["min_logprob"])[:10]:
            print(record)

    stratified_order = build_stratified_index_order(
        record_categories,
        EFFECTIVE_BATCH_SIZE,
        EPOCH_BATCH_ORDER_SEED,
    )

    training_args = TrainingArguments(
        output_dir="/kaggle/working/sft_output",
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="constant",
        warmup_steps=0,
        optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch",
        bf16=True,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
        seed=SEED,
        dataloader_num_workers=2,
        remove_unused_columns=False,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        weight_decay=0.0,
        adam_beta1=0.9,
        adam_beta2=0.95,
        adam_epsilon=1e-8,
        max_grad_norm=1e9,
    )

    trainer = StratifiedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=MaskedDataCollator(tokenizer.pad_token_id),
        stratified_order=stratified_order,
        loss_config=loss_config,
        lr_schedule_config=lr_schedule_config,
    )

    train_result = trainer.train()
    print(train_result)

    ADAPTER_DIR = "/kaggle/working/sft_adapter"
    os.makedirs(ADAPTER_DIR, exist_ok=True)
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    print(f"Saved adapter to {ADAPTER_DIR}")

    if RUN_LOGPROB_ANALYSIS:
        log_name = datetime.now().strftime("%m-%d-%H-%M")
        log_root = Path(TRAINING_LOGS_ROOT) / log_name
        log_root.mkdir(parents=True, exist_ok=True)
        records_to_analyze = records
        if LOGPROB_ANALYSIS_LIMIT is not None:
            records_to_analyze = records_to_analyze[:LOGPROB_ANALYSIS_LIMIT]
        collect_logprob_logs(records_to_analyze, log_root)
else:
    print("USE_PRETRAINED=1: skipping training cell.")


## Mode B: Load Pre-trained LoRA

In [9]:
if USE_PRETRAINED:
    import os

    SRC_ADAPTER_DIR = PRETRAINED_ADAPTER_DATASET_PATH
    required_files = ["adapter_config.json", "adapter_model.safetensors"]

    print("Using pre-trained adapter from:", SRC_ADAPTER_DIR)
    for fname in required_files:
        fpath = os.path.join(SRC_ADAPTER_DIR, fname)
        if not os.path.exists(fpath):
            raise FileNotFoundError(f"Missing required adapter file: {fpath}")
        print(f"  {fname}: {os.path.getsize(fpath)/1024/1024:.1f} MB")
else:
    print("TRAIN_ON_KAGGLE=1: pretrained adapter path check skipped.")


TRAIN_ON_KAGGLE=1: pretrained adapter path check skipped.


## Create submission.zip

In [10]:
import json, os, shutil, zipfile

OUTPUT_DIR = "/kaggle/working"
SUBMISSION_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "submission_adapter")
os.makedirs(SUBMISSION_ADAPTER_DIR, exist_ok=True)

required_files = ["adapter_config.json", "adapter_model.safetensors"]

if TRAIN_ON_KAGGLE:
    src_adapter_dir = "/kaggle/working/sft_adapter"
    print("Packaging freshly trained adapter from:", src_adapter_dir)
else:
    src_adapter_dir = PRETRAINED_ADAPTER_DATASET_PATH
    print("Packaging pre-trained adapter directly from:", src_adapter_dir)

for fname in required_files:
    src = os.path.join(src_adapter_dir, fname)
    dst = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing required adapter file: {src}")
    shutil.copy2(src, dst)
    print(f"Copied {fname} ({os.path.getsize(dst)/1024/1024:.1f} MB)")

config_path = os.path.join(SUBMISSION_ADAPTER_DIR, "adapter_config.json")
with open(config_path, "r") as f:
    cfg = json.load(f)

cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"] = 0.0

with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        fpath = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
        zf.write(fpath, fname)
        print(f"  Added {fname}")

zip_sz = os.path.getsize(zip_path) / 1024 / 1024
print(f"\nsubmission.zip: {zip_sz:.1f} MB")
print("Done! Ready to submit.")


Packaging freshly trained adapter from: /kaggle/working/sft_adapter
Copied adapter_config.json (0.0 MB)
Copied adapter_model.safetensors (4061.8 MB)
  Added adapter_config.json
  Added adapter_model.safetensors

submission.zip: 3638.5 MB
Done! Ready to submit.
